In [1]:
import os
import sys
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    confusion_matrix,
    roc_auc_score,
    f1_score,
    recall_score,
    precision_score,
)
from imblearn.over_sampling import SMOTE

# Add project root
ROOT_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT_DIR not in sys.path:
    sys.path.append(ROOT_DIR)

from src.data_enrichment import get_features

# Load enriched dataset
df_feats, feature_cols = get_features("../data/raw")

df_ml = df_feats[
    (df_feats["season_end_year"] >= 2008) &
    (df_feats["season_end_year"] <= 2024) &
    (df_feats["minutes_played"] >= 100)
].copy()

df_ml.shape

(23442, 79)

In [2]:
df_train_full = df_ml[df_ml["season_end_year"] <= 2022].copy()
df_test       = df_ml[df_ml["season_end_year"] >= 2023].copy()

print("Train FULL (2008–2022):", df_train_full.shape)
print("Test (2023–2024):      ", df_test.shape)

X_train_full = df_train_full[feature_cols]
y_train_full = df_train_full["ballon_dor_winner"].astype(int)

X_test = df_test[feature_cols]
y_test = df_test["ballon_dor_winner"].astype(int)


Train FULL (2008–2022): (20339, 79)
Test (2023–2024):       (3103, 79)


In [3]:
print("Before SMOTE (train_full):")
print(y_train_full.value_counts())

sm = SMOTE(k_neighbors=1, random_state=42)
X_train_full_res, y_train_full_res = sm.fit_resample(X_train_full, y_train_full)

print("\nAfter SMOTE (train_full):")
print(y_train_full_res.value_counts())
X_train_full_res.shape


Before SMOTE (train_full):
ballon_dor_winner
0    20325
1       14
Name: count, dtype: int64

After SMOTE (train_full):
ballon_dor_winner
0    20325
1    20325
Name: count, dtype: int64


(40650, 72)

In [4]:
scaler_final = StandardScaler()
X_train_full_res_scaled = scaler_final.fit_transform(X_train_full_res)
X_test_scaled = scaler_final.transform(X_test)


In [5]:
best_params = {
'bootstrap': False,
 'max_depth': 25,
 'max_features': 'log2',
 'min_samples_leaf': 1,
 'min_samples_split': 10,
 'n_estimators': 1447
}

rf_final = RandomForestClassifier(**best_params)
rf_final


,n_estimators,1447
,criterion,'gini'
,max_depth,25
,min_samples_split,10
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'log2'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,False
,oob_score,False


In [6]:
rf_final.fit(X_train_full_res_scaled, y_train_full_res)

,n_estimators,1447
,criterion,'gini'
,max_depth,25
,min_samples_split,10
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'log2'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,False
,oob_score,False


In [7]:
best_threshold = 0.162539  

proba_test = rf_final.predict_proba(X_test_scaled)[:, 1]
pred_test_default = (proba_test >= 0.5).astype(int)
pred_test_opt     = (proba_test >= best_threshold).astype(int)

print("=== TEST RESULTS — threshold 0.50 (default) ===")
print("AUC:", roc_auc_score(y_test, proba_test))
print("Recall:", recall_score(y_test, pred_test_default))
print("Precision:", precision_score(y_test, pred_test_default))
print("F1:", f1_score(y_test, pred_test_default))
print("Confusion matrix:\n", confusion_matrix(y_test, pred_test_default))

print("\n=== TEST RESULTS — threshold", round(best_threshold, 3), "(optimal) ===")
print("AUC:", roc_auc_score(y_test, proba_test))
print("Recall:", recall_score(y_test, pred_test_opt))
print("Precision:", precision_score(y_test, pred_test_opt))
print("F1:", f1_score(y_test, pred_test_opt))
print("Confusion matrix:\n", confusion_matrix(y_test, pred_test_opt))


=== TEST RESULTS — threshold 0.50 (default) ===
AUC: 0.7040470815865848
Recall: 0.0
Precision: 0.0
F1: 0.0
Confusion matrix:
 [[3101    0]
 [   2    0]]

=== TEST RESULTS — threshold 0.163 (optimal) ===
AUC: 0.7040470815865848
Recall: 0.0
Precision: 0.0
F1: 0.0
Confusion matrix:
 [[3101    0]
 [   2    0]]


c:\Users\leodo\OneDrive\Escritorio\machine learning\Machine-learning\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\leodo\OneDrive\Escritorio\machine learning\Machine-learning\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [8]:
from joblib import dump

dump(
    {
        "model": rf_final,
        "scaler": scaler_final,
        "feature_cols": feature_cols,
        "threshold": best_threshold
    },
    "rf_final_2008_2022.pkl"
)

print("Final model saved as rf_final_2008_2022.pkl")


Final model saved as rf_final_2008_2022.pkl
